In [ ]:
import pandas as pd

# ---------------------------------------------
# 1. تعریف نام ستون‌ها طبق توضیح صورت‌مسئله
# ---------------------------------------------
column_names = [
    "id", "age", "gender", "height", "weight",
    "ap_hi", "ap_lo", "cholesterol", "gluc",
    "smoke", "alco", "active", "cardio"
]

# ---------------------------------------------
# 2. خواندن دیتاست با تعیین جداکننده ';'
#    اگر sep را تعیین نکنیم، همهٔ داده‌ها
#    در یک ستون قرار می‌گیرند.
# ---------------------------------------------
df = pd.read_csv("CA2_cardio_dataset.csv", sep=';', header=0, names=column_names)

# ---------------------------------------------
# 3. نمایش ۵ ردیف اول برای بررسی صحت داده‌ها
# ---------------------------------------------
print("نمایش ۵ ردیف اول دیتاست:")
print(df.head(), "\n")

# ---------------------------------------------
# 4. بررسی ساختار کلی دیتافریم
#    شامل نوع داده‌ها، تعداد ستون‌ها،
#    تعداد مقادیر non-null و …
# ---------------------------------------------
print("اطلاعات کلی دیتاست (df.info()):")
print(df.info(), "\n")

# ---------------------------------------------
# 5. آمار توصیفی ویژگی‌های عددی
# ---------------------------------------------
print("خلاصه آمار توصیفی (df.describe()):")
print(df.describe(), "\n")

# ---------------------------------------------
# 6. بررسی توزیع برچسب هدف (cardio)
#    این ستون کلاس 0 و 1 دارد (بیمار / سالم)
# ---------------------------------------------
print("توزیع مقادیر برچسب 'cardio':")
print(df["cardio"].value_counts())


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------
# 1. حذف ستون id
# ---------------------------------------------
df = df.drop(columns=["id"])

# ---------------------------------------------
# 2. حذف رکوردهای غیرعادی (Outliers)
#    مقادیر منفی یا بیش از حد بزرگ در ap_hi و ap_lo
# ---------------------------------------------

# شرط‌های منطقی فشار خون:
condition = (
    (df["ap_hi"] >= 80) & (df["ap_hi"] <= 240) &
    (df["ap_lo"] >= 50) & (df["ap_lo"] <= 200) &
    (df["ap_hi"] >= df["ap_lo"])
)

# تعداد رکوردهای حذف‌شده
removed_count = len(df) - sum(condition)

print(f"تعداد داده‌های حذف شده به دلیل فشار خون غیرواقعی: {removed_count}")

# اعمال فیلتر
df = df[condition]

# ---------------------------------------------
# 3. نرمال‌سازی ویژگی‌ها (به جز ستون برچسب cardio)
# ---------------------------------------------
features = df.drop(columns=["cardio"])
labels = df["cardio"]

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# تبدیل دوباره به DataFrame برای راحتی
df_scaled = pd.DataFrame(features_scaled, columns=features.columns)
df_scaled["cardio"] = labels.values

print("\nنمونه‌ای از داده‌های نرمال‌سازی شده:")
print(df_scaled.head())


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# ---------------------------------------------
# 1. جداسازی ویژگی‌ها (X) و برچسب (y)
# ---------------------------------------------
X = df_scaled.drop(columns=["cardio"]).values
y = df_scaled["cardio"].values.reshape(-1, 1) # تبدیل به ستون برای محاسبات

# ---------------------------------------------
# 2. تقسیم به آموزش (80%) و تست (20%)
# ---------------------------------------------
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------
# 3. Transpose کردن ماتریس‌ها برای محاسبات شبکه عصبی
#    ما می‌خواهیم هر ستون یک نمونه باشد:
#    X_train باید ابعاد (11, m) داشته باشد.
# ---------------------------------------------
X_train = X_train_raw.T
y_train = y_train_raw.T
X_test = X_test_raw.T
y_test = y_test_raw.T

print(f"ابعاد داده‌های آموزش: {X_train.shape}") # خروجی باید (11, 54920) باشد
print(f"ابعاد برچسب‌های آموزش: {y_train.shape}")
print(f"ابعاد داده‌های تست: {X_test.shape}")


In [ ]:
def sigmoid(z):
    """
    تابع سیگموئید برای لایه خروجی
    """
    return 1 / (1 + np.exp(-z))

def relu(z):
    """
    تابع ReLU برای لایه‌های پنهان
    """
    return np.maximum(0, z)

def relu_backward(dA, z):
    """
    مشتق تابع ReLU برای استفاده در Backpropagation
    """
    dZ = np.array(dA, copy=True)
    dZ[z <= 0] = 0
    return dZ

def sigmoid_backward(dA, z):
    """
    مشتق تابع Sigmoid برای استفاده در Backpropagation
    """
    s = sigmoid(z)
    dZ = dA * s * (1 - s)
    return dZ
# تست تابع سیگموید
print(sigmoid(np.array([0, 2, -2])))



In [ ]:
import numpy as np

# ---------------------------------------------
# مقداردهی اولیه وزن‌ها و بایاس
# ---------------------------------------------
np.random.seed(42)

n_features = X_train.shape[0]

W = np.random.randn(n_features, 1) * 0.01
b = 0

# ---------------------------------------------
# Forward Propagation
# ---------------------------------------------
def forward_propagation(X, W, b):
    """
    X: (n_features, m)
    W: (n_features, 1)
    b: scalar
    """
    Z = np.dot(W.T, X) + b      # (1, m)
    A = sigmoid(Z)              # (1, m)
    return A

# اجرای forward روی داده‌های آموزش
A_train = forward_propagation(X_train, W, b)

# ---------------------------------------------
# بررسی خروجی‌ها
# ---------------------------------------------
print("شکل A_train:", A_train.shape)
print("۵ مقدار اول خروجی شبکه:")
print(A_train[0, :5])


In [ ]:
# ---------------------------------------------
# محاسبه Cross-Entropy Loss
# ---------------------------------------------

def compute_loss(A, Y):
    """
    A: پیش‌بینی شبکه (1, m)
    Y: برچسب واقعی (1, m)
    """
    
    m = Y.shape[1]
    
    loss = -(1/m) * np.sum(
        Y * np.log(A) + (1 - Y) * np.log(1 - A)
    )
    
    return loss


# محاسبه loss برای داده‌های آموزش
loss = compute_loss(A_train, y_train)

print("Loss اولیه شبکه:")
print(loss)


In [ ]:
# ---------------------------------------------
# Backpropagation
# ---------------------------------------------

def backward_propagation(X, A, Y):
    """
    X : (n_features, m)
    A : (1, m)
    Y : (1, m)
    """
    m = X.shape[1]

    # محاسبه dZ
    dZ = A - Y                       # (1, m)

    # محاسبه dW
    dW = (1/m) * np.dot(X, dZ.T)     # (n_features, 1)

    # محاسبه db
    db = (1/m) * np.sum(dZ)          # scalar

    return dW, db


# تست اولیه:
dW, db = backward_propagation(X_train, A_train, y_train)

print("شکل dW:", dW.shape)
print("db:", db)
print("۵ مقدار اول dW:")
print(dW[:5])


In [ ]:
# ---------------------------------------------
# گام ۸: آموزش مدل با Gradient Descent
# ---------------------------------------------

def train_network(X, Y, learning_rate=0.01, epochs=1000):
    n_features = X.shape[0]
    m = X.shape[1]
    
    # مقداردهی اولیه وزن‌ها
    W = np.random.randn(n_features, 1) * 0.01
    b = 0
    
    losses = []
    
    for i in range(epochs):
        
        # Forward
        A = forward_propagation(X, W, b)
        
        # Loss
        loss = compute_loss(A, Y)
        losses.append(loss)
        
        # Backward
        dW, db = backward_propagation(X, A, Y)
        
        # Update
        W = W - learning_rate * dW
        b = b - learning_rate * db
        
        # نمایش دوره‌ای Loss
        if i % 100 == 0:
            print(f"Epoch {i}, Loss: {loss}")
    
    return W, b, losses


# اجرای آموزش
W_trained, b_trained, losses = train_network(
    X_train, y_train, 
    learning_rate=0.01, 
    epochs=1000
)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(losses, label="Training Loss", color='blue')
plt.xlabel("Epoch")
plt.ylabel("Loss (Cross-Entropy)")
plt.title("Loss Curve During Training")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# ---------------------------------------------
# گام ۱۰: توابع پیش‌بینی و محاسبه دقت
# ---------------------------------------------

def predict(X, W, b):
    # خروجی خام مدل (Probability)
    probabilities = forward_propagation(X, W, b)
    
    # تبدیل به 0 و 1 (Threshold = 0.5)
    predictions = (probabilities > 0.5).astype(int)
    
    return predictions

def calculate_accuracy(y_true, y_pred):
    # محاسبه تعداد موارد صحیح تقسیم بر کل
    accuracy = np.mean(y_true == y_pred) * 100
    return accuracy

# محاسبه پیش‌بینی‌ها
y_pred_train = predict(X_train, W_trained, b_trained)
y_pred_test = predict(X_test, W_trained, b_trained)

# محاسبه Accuracy
acc_train = calculate_accuracy(y_train, y_pred_train)
acc_test = calculate_accuracy(y_test, y_pred_test)

print(f"Accuracy on Train set: {acc_train:.2f}%")
print(f"Accuracy on Test set: {acc_test:.2f}%")


In [ ]:
def initialize_parameters_deep(n_x, n_h, n_y):
    """
    n_x: تعداد ویژگی‌های ورودی (11)
    n_h: تعداد نورون‌های لایه مخفی (مثلا 4)
    n_y: تعداد نورون‌های خروجی (1)
    """
    np.random.seed(42) # برای نتایج مشابه
    
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    
    parameters = {
        "W1": W1, "b1": b1,
        "W2": W2, "b2": b2
    }
    return parameters

# تست با 4 نورون در لایه مخفی
params = initialize_parameters_deep(11, 4, 1)
print("W1 shape:", params["W1"].shape) # باید (4, 11) باشد
print("W2 shape:", params["W2"].shape) # باید (1, 4) باشد


In [ ]:
import numpy as np

def sigmoid(z):
    """
    تابع سیگموید:
    ورودی:
        z : عدد، بردار یا ماتریس
    خروجی:
        s : مقدار سیگموید عنصر به عنصر
    """
    s = 1 / (1 + np.exp(-z))
    return s

def tanh(z):
    """
    تابع تان‌هَش (tanh):
    معمولاً برای لایه‌های مخفی استفاده می‌شود.
    """
    return np.tanh(z)


In [ ]:
def initialize_parameters_deep(n_x, n_h, n_y, seed=42):
    """
    مقداردهی اولیه پارامترهای شبکه عصبی دو لایه:
        لایه ورودی -> لایه مخفی -> لایه خروجی
    
    ورودی‌ها:
        n_x : تعداد نودهای لایه ورودی (تعداد ویژگی‌ها)
        n_h : تعداد نورون‌های لایه مخفی
        n_y : تعداد نورون‌های لایه خروجی (برای باینری = 1)
        seed: برای تکرارپذیری نتایج (random seed)
        
    خروجی:
        parameters : دیکشنری شامل
            W1 با ابعاد (n_h, n_x)
            b1 با ابعاد (n_h, 1)
            W2 با ابعاد (n_y, n_h)
            b2 با ابعاد (n_y, 1)
    """
    np.random.seed(seed)
    
    # وزن‌ها را کوچک مقداردهی می‌کنیم تا خروجی‌ها در ابتدا خیلی بزرگ نشوند
    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))
    
    parameters = {
        "W1": W1, "b1": b1,
        "W2": W2, "b2": b2
    }
    return parameters

# تست اولیه
n_x = X_train.shape[0]   # 11 ویژگی (بعد از حذف id و جدا کردن y)
n_h = 4                  # تعداد نورون‌های لایه مخفی (قابل تغییر)
n_y = 1                  # خروجی باینری

params = initialize_parameters_deep(n_x, n_h, n_y)
print("W1 shape:", params["W1"].shape)  # (n_h, n_x)
print("b1 shape:", params["b1"].shape)  # (n_h, 1)
print("W2 shape:", params["W2"].shape)  # (n_y, n_h)
print("b2 shape:", params["b2"].shape)  # (n_y, 1)


In [ ]:
def forward_propagation_deep(X, parameters):
    """
    انجام پروپاگیشن رو به جلو در شبکه ۲ لایه‌ای:
        X -> [Linear -> Tanh] -> [Linear -> Sigmoid] -> Y_hat
    
    ورودی‌ها:
        X          : داده‌های ورودی با ابعاد (n_x, m)
                     n_x = تعداد ویژگی‌ها
                     m   = تعداد نمونه‌ها
        parameters : دیکشنری شامل W1, b1, W2, b2
    
    خروجی:
        A2     : خروجی نهایی شبکه (احتمال کلاس 1) با ابعاد (1, m)
        cache  : دیکشنری حاوی مقادیر میانی جهت استفاده در Backward:
                 Z1, A1, Z2, A2
    """
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # لایه مخفی: Linear -> Tanh
    Z1 = np.dot(W1, X) + b1          # (n_h, m)
    A1 = tanh(Z1)                    # (n_h, m)
    
    # لایه خروجی: Linear -> Sigmoid
    Z2 = np.dot(W2, A1) + b2         # (1, m)
    A2 = sigmoid(Z2)                 # (1, m)
    
    cache = {
        "Z1": Z1, "A1": A1,
        "Z2": Z2, "A2": A2
    }
    
    return A2, cache


In [ ]:
def compute_loss(Y_hat, Y):
    """
    محاسبه تابع هزینه Cross-Entropy برای طبقه‌بندی دودویی.
    
    ورودی‌ها:
        Y_hat : خروجی مدل (احتمالات) با ابعاد (1, m)
        Y     : برچسب‌های واقعی با ابعاد (1, m)
    
    خروجی:
        loss  : مقدار اسکالر تابع هزینه
    """
    m = Y.shape[1]
    
    # جلوگیری از log(0) با کلیپ کردن
    epsilon = 1e-8
    Y_hat_clipped = np.clip(Y_hat, epsilon, 1 - epsilon)
    
    loss = - (1/m) * np.sum(Y * np.log(Y_hat_clipped) + (1 - Y) * np.log(1 - Y_hat_clipped))
    return loss


In [ ]:
def backward_propagation_deep(X, Y, parameters, cache):
    """
    انجام پروپاگیشن رو به عقب (Backpropagation) برای شبکه ۲ لایه‌ای.
    
    ورودی‌ها:
        X          : ورودی‌ها با ابعاد (n_x, m)
        Y          : برچسب‌های واقعی با ابعاد (1, m)
        parameters : دیکشنری شامل W1, b1, W2, b2
        cache      : دیکشنری شامل Z1, A1, Z2, A2 از مرحله Forward
    
    خروجی:
        grads : دیکشنری شامل گرادیان‌ها:
                dW1, db1, dW2, db2
    """
    m = X.shape[1]
    
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    
    A1 = cache["A1"]
    A2 = cache["A2"]
    
    # گرادیان لایه خروجی
    dZ2 = A2 - Y                               # (1, m)
    dW2 = (1/m) * np.dot(dZ2, A1.T)           # (1, n_h)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)  # (1, 1)
    
    # گرادیان لایه مخفی
    dA1 = np.dot(W2.T, dZ2)                   # (n_h, m)
    dZ1 = dA1 * (1 - np.power(A1, 2))         # مشتق tanh
    dW1 = (1/m) * np.dot(dZ1, X.T)           # (n_h, n_x)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)  # (n_h, 1)
    
    grads = {
        "dW1": dW1, "db1": db1,
        "dW2": dW2, "db2": db2
    }
    return grads


In [ ]:
def train_network_deep(X, Y, n_h, num_epochs=1000, learning_rate=0.01, print_loss=True):
    """
    آموزش شبکه عصبی ۲ لایه‌ای با استفاده از Gradient Descent.
    
    ورودی‌ها:
        X            : داده‌های ورودی با ابعاد (n_x, m)
        Y            : برچسب‌های واقعی با ابعاد (1, m)
        n_h          : تعداد نورون‌های لایه مخفی
        num_epochs   : تعداد epochهای آموزش
        learning_rate: نرخ یادگیری گرادیان نزولی
        print_loss   : اگر True باشد هر 100 epoch مقدار Loss چاپ می‌شود
    
    خروجی:
        parameters   : پارامترهای نهایی آموزش‌دیده (W1, b1, W2, b2)
        losses       : لیست مقادیر Loss در طول آموزش برای رسم نمودار
    """
    n_x = X.shape[0]
    n_y = 1
    
    parameters = initialize_parameters_deep(n_x, n_h, n_y)
    losses = []
    
    for epoch in range(num_epochs):
        # 1) Forward
        Y_hat, cache = forward_propagation_deep(X, parameters)
        
        # 2) Loss
        loss = compute_loss(Y_hat, Y)
        losses.append(loss)
        
        # 3) Backward
        grads = backward_propagation_deep(X, Y, parameters, cache)
        
        # 4) به‌روزرسانی پارامترها با گرادیان نزولی
        parameters["W1"] -= learning_rate * grads["dW1"]
        parameters["b1"] -= learning_rate * grads["db1"]
        parameters["W2"] -= learning_rate * grads["dW2"]
        parameters["b2"] -= learning_rate * grads["db2"]
        
        # چاپ مقدار Loss در برخی epochها برای نظارت بر همگرایی
        if print_loss and epoch % 100 == 0:
            print(f"Epoch {epoch} / {num_epochs} - Loss: {loss:.4f}")
    
    return parameters, losses


In [ ]:
def predict_deep(X, parameters, threshold=0.5):
    """
    پیش‌بینی کلاس (0 یا 1) با استفاده از شبکه ۲ لایه‌ای.
    """
    Y_hat, _ = forward_propagation_deep(X, parameters)
    Y_pred = (Y_hat >= threshold).astype(int)
    return Y_hat, Y_pred

def calculate_accuracy(Y_pred, Y_true):
    """
    محاسبه دقت (Accuracy) مدل.
    """
    return np.mean(Y_pred == Y_true) * 100


In [ ]:
# تعداد نورون‌های لایه مخفی را می‌توانی تغییر دهی (برای بند ۶ پروژه خیلی مهم است)
n_h = 4
num_epochs = 1000
learning_rate = 0.01

parameters_deep, losses_deep = train_network_deep(
    X_train, y_train,
    n_h=n_h,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    print_loss=True
)

# دقت روی داده‌های آموزش
_, y_train_pred = predict_deep(X_train, parameters_deep)
train_acc = calculate_accuracy(y_train_pred, y_train)
print(f"Train Accuracy (n_h={n_h}): {train_acc:.2f}%")

# دقت روی داده‌های تست
_, y_test_pred = predict_deep(X_test, parameters_deep)
test_acc = calculate_accuracy(y_test_pred, y_test)
print(f"Test Accuracy (n_h={n_h}): {test_acc:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(losses_deep)
plt.xlabel("Epoch")
plt.ylabel("Loss (Cross-Entropy)")
plt.title(f"Training Loss - 2-layer NN (n_h = {n_h})")
plt.grid(True)
plt.show()


In [ ]:
_, y_train_pred = predict_deep(X_train, parameters_deep)
train_acc = calculate_accuracy(y_train_pred, y_train)
print("Train Accuracy:", train_acc)

_, y_test_pred = predict_deep(X_test, parameters_deep)
test_acc = calculate_accuracy(y_test_pred, y_test)
print("Test Accuracy:", test_acc)


In [ ]:
hidden_sizes = [2, 4, 8, 16, 32]

results = []

for n_h in hidden_sizes:
    print(f"\nTraining model with n_h = {n_h}")
    parameters, losses = train_network_deep(
        X_train, y_train,
        n_h=n_h,
        num_epochs=800,
        learning_rate=0.01,
        print_loss=False
    )
    
    _, train_pred = predict_deep(X_train, parameters)
    _, test_pred  = predict_deep(X_test, parameters)
    
    train_acc = calculate_accuracy(train_pred, y_train)
    test_acc  = calculate_accuracy(test_pred, y_test)
    
    results.append((n_h, train_acc, test_acc))

print("\nResults:")
for r in results:
    print(f"n_h={r[0]}  --> Train={r[1]:.2f}%   Test={r[2]:.2f}%")


In [ ]:
# Train deep neural network with n_h = 16
n_h = 16

parameters_16, losses_16 = train_network_deep(
    X_train,
    y_train,
    n_h=n_h,
    num_epochs=10000,
    learning_rate=0.01,
    print_loss=True   # می‌خواهی نمودار loss ببینی یا نه، قابل تنظیم است
)

# Evaluate accuracy
_, y_train_pred_16 = predict_deep(X_train, parameters_16)
train_acc_16 = calculate_accuracy(y_train_pred_16, y_train)

_, y_test_pred_16 = predict_deep(X_test, parameters_16)
test_acc_16 = calculate_accuracy(y_test_pred_16, y_test)

print("\n---- Results for n_h = 16 ----")
print("Train Accuracy:", train_acc_16)
print("Test Accuracy:", test_acc_16)


In [ ]:
plt.plot(losses_16)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve (n_h = 16)")
plt.show()


In [ ]:
def initialize_parameters_3layer(n_x, n_h1, n_h2, n_y):
    np.random.seed(42)

    W1 = np.random.randn(n_h1, n_x) * 0.01
    b1 = np.zeros((n_h1, 1))

    W2 = np.random.randn(n_h2, n_h1) * 0.01
    b2 = np.zeros((n_h2, 1))

    W3 = np.random.randn(n_y, n_h2) * 0.01
    b3 = np.zeros((n_y, 1))

    return {"W1": W1, "b1": b1,
            "W2": W2, "b2": b2,
            "W3": W3, "b3": b3}


In [ ]:
def forward_propagation_3layer(X, parameters):
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    W3, b3 = parameters["W3"], parameters["b3"]

    Z1 = np.dot(W1, X) + b1
    A1 = np.tanh(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = np.tanh(Z2)

    Z3 = np.dot(W3, A2) + b3
    A3 = 1 / (1 + np.exp(-Z3))  # sigmoid

    cache = (Z1, A1, Z2, A2, Z3, A3)
    return A3, cache


In [ ]:
def backward_propagation_3layer(X, Y, parameters, cache):
    m = X.shape[1]

    W1, W2, W3 = parameters["W1"], parameters["W2"], parameters["W3"]
    Z1, A1, Z2, A2, Z3, A3 = cache

    dZ3 = A3 - Y
    dW3 = (1 / m) * np.dot(dZ3, A2.T)
    db3 = (1 / m) * np.sum(dZ3, axis=1, keepdims=True)

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = dA2 * (1 - np.tanh(Z2) ** 2)
    dW2 = (1 / m) * np.dot(dZ2, A1.T)
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * (1 - np.tanh(Z1) ** 2)
    dW1 = (1 / m) * np.dot(dZ1, X.T)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {"dW1": dW1, "db1": db1,
             "dW2": dW2, "db2": db2,
             "dW3": dW3, "db3": db3}
    return grads


In [ ]:
def train_network_3layer(X, Y, n_h1=16, n_h2=16, num_epochs=10000, learning_rate=0.01, print_loss=True):
    n_x = X.shape[0]
    n_y = 1

    parameters = initialize_parameters_3layer(n_x, n_h1, n_h2, n_y)
    losses = []

    for i in range(num_epochs):
        A3, cache = forward_propagation_3layer(X, parameters)
        loss = compute_loss(A3, Y)
        grads = backward_propagation_3layer(X, Y, parameters, cache)

        parameters["W1"] -= learning_rate * grads["dW1"]
        parameters["b1"] -= learning_rate * grads["db1"]
        parameters["W2"] -= learning_rate * grads["dW2"]
        parameters["b2"] -= learning_rate * grads["db2"]
        parameters["W3"] -= learning_rate * grads["dW3"]
        parameters["b3"] -= learning_rate * grads["db3"]

        if i % 100 == 0:
            losses.append(loss)
            if print_loss:
                print(f"Epoch {i} Loss: {loss}")

    return parameters, losses


In [ ]:
def predict_3layer(X, parameters):
    A3, _ = forward_propagation_3layer(X, parameters)
    Y_pred = (A3 > 0.5).astype(int)
    return A3, Y_pred


In [ ]:
parameters_3, losses_3 = train_network_3layer(
    X_train,
    y_train,
    n_h1=16,
    n_h2=16,
    num_epochs=10000,
    learning_rate=0.01,
    print_loss=False
)

_, train_pred = predict_3layer(X_train, parameters_3)
_, test_pred = predict_3layer(X_test, parameters_3)

print("Train Accuracy:", calculate_accuracy(train_pred, y_train))
print("Test Accuracy:", calculate_accuracy(test_pred, y_test))


In [ ]:
plt.plot(losses_3)
plt.title("Loss Curve (3-layer, 16 neurons, 10000 epochs)")
plt.xlabel("Epoch (x100)")
plt.ylabel("Loss")
plt.show()


In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

def xavier_init(size_in, size_out):
    return np.random.randn(size_out, size_in) * np.sqrt(2 / (size_in + size_out))


In [ ]:
def initialize_parameters_3layer_optimized(n_x, n_h1, n_h2, n_y):
    np.random.seed(42)

    W1 = xavier_init(n_x, n_h1)
    b1 = np.zeros((n_h1, 1))

    W2 = xavier_init(n_h1, n_h2)
    b2 = np.zeros((n_h2, 1))

    W3 = xavier_init(n_h2, n_y)
    b3 = np.zeros((n_y, 1))

    return {"W1": W1, "b1": b1,
            "W2": W2, "b2": b2,
            "W3": W3, "b3": b3}


In [ ]:
def forward_propagation_optimized(X, parameters):
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    W3, b3 = parameters["W3"], parameters["b3"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)

    Z3 = np.dot(W3, A2) + b3
    A3 = 1 / (1 + np.exp(-Z3))  # sigmoid

    cache = (Z1, A1, Z2, A2, Z3, A3)
    return A3, cache


In [ ]:
def backward_propagation_optimized(X, Y, parameters, cache):
    m = X.shape[1]

    W1, W2, W3 = parameters["W1"], parameters["W2"], parameters["W3"]
    Z1, A1, Z2, A2, Z3, A3 = cache

    dZ3 = A3 - Y
    dW3 = (1/m) * np.dot(dZ3, A2.T)
    db3 = (1/m) * np.sum(dZ3, axis=1, keepdims=True)

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {"dW1": dW1, "db1": db1,
             "dW2": dW2, "db2": db2,
             "dW3": dW3, "db3": db3}
    return grads


In [ ]:
def initialize_adam(parameters):
    v = {}
    s = {}
    for key in parameters:
        v["v_" + key] = np.zeros_like(parameters[key])
        s["s_" + key] = np.zeros_like(parameters[key])
    return v, s


In [ ]:
def train_network_3layer_optimized(X, Y, n_h1=16, n_h2=16,
                                   num_epochs=10000, learning_rate=0.005,
                                   beta1=0.9, beta2=0.999, epsilon=1e-8):
    
    n_x = X.shape[0]
    n_y = 1

    parameters = initialize_parameters_3layer_optimized(n_x, n_h1, n_h2, n_y)
    v, s = initialize_adam(parameters)

    losses = []
    t = 0

    for i in range(num_epochs):
        A3, cache = forward_propagation_optimized(X, parameters)
        loss = compute_loss(A3, Y)
        grads = backward_propagation_optimized(X, Y, parameters, cache)

        t += 1

        for key in parameters:
            # Adam update
            v["v_" + key] = beta1 * v["v_" + key] + (1 - beta1) * grads["d" + key]
            s["s_" + key] = beta2 * s["s_" + key] + (1 - beta2) * (grads["d" + key] ** 2)

            v_corrected = v["v_" + key] / (1 - beta1**t)
            s_corrected = s["s_" + key] / (1 - beta2**t)

            parameters[key] -= learning_rate * v_corrected / (np.sqrt(s_corrected) + epsilon)

        if i % 100 == 0:
            losses.append(loss)
            print(f"Epoch {i} Loss: {loss}")

    return parameters, losses


In [ ]:
def predict_optimized(X, parameters):
    A3, _ = forward_propagation_optimized(X, parameters)
    Y_pred = (A3 > 0.5).astype(int)
    return A3, Y_pred


In [ ]:
parameters_opt, losses_opt = train_network_3layer_optimized(
    X_train,
    y_train,
    n_h1=16,
    n_h2=16,
    num_epochs=10000,
    learning_rate=0.005
)

_, train_pred = predict_optimized(X_train, parameters_opt)
_, test_pred = predict_optimized(X_test, parameters_opt)

print("\n---- Results for n_h = 16 ----")
print("Train Accuracy:", calculate_accuracy(train_pred, y_train))
print("Test Accuracy:", calculate_accuracy(test_pred, y_test))


In [ ]:
# ------------------------------
# توابع پایه برای شبکه عصبی ۳ لایه
# شامل: ReLU ، مشتق ReLU ، تابع فعال‌سازی سیگموید
# و مقداردهی اولیه به روش Xavier
# ------------------------------

import numpy as np

def relu(Z):
    # تابع فعال‌سازی ReLU = max(0, Z)
    return np.maximum(0, Z)

def relu_derivative(Z):
    # مشتق ReLU برای Backpropagation
    return (Z > 0).astype(float)

def xavier_init(size_in, size_out):
    # مقداردهی اولیه وزن‌ها با روش Xavier
    return np.random.randn(size_out, size_in) * np.sqrt(2 / (size_in + size_out))

def initialize_parameters_3layer(n_x, n_h1, n_h2, n_y):
    # مقداردهی اولیه پارامترهای شبکه ۳ لایه با Xavier
    np.random.seed(42)

    W1 = xavier_init(n_x, n_h1)
    b1 = np.zeros((n_h1, 1))

    W2 = xavier_init(n_h1, n_h2)
    b2 = np.zeros((n_h2, 1))

    W3 = xavier_init(n_h2, n_y)
    b3 = np.zeros((n_y, 1))

    return {"W1": W1, "b1": b1,
            "W2": W2, "b2": b2,
            "W3": W3, "b3": b3}

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))


# ------------------------------
# Forward Propagation برای شبکه ۳ لایه
# شامل ReLU در لایه‌های مخفی و Sigmoid در خروجی
# ------------------------------
def forward_propagation(X, parameters):
    W1, b1 = parameters["W1"], parameters["b1"]
    W2, b2 = parameters["W2"], parameters["b2"]
    W3, b3 = parameters["W3"], parameters["b3"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)

    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    cache = (Z1, A1, Z2, A2, Z3, A3)
    return A3, cache


# ------------------------------
# Backpropagation برای شبکه ۳ لایه
# استفاده از مشتق ReLU و مشتق تابع سیگموید
# ------------------------------
def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]

    W1, W2, W3 = parameters["W1"], parameters["W2"], parameters["W3"]
    Z1, A1, Z2, A2, Z3, A3 = cache

    dZ3 = A3 - Y
    dW3 = (1/m) * np.dot(dZ3, A2.T)
    db3 = (1/m) * np.sum(dZ3, axis=1, keepdims=True)

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = (1/m) * np.dot(dZ2, A1.T)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (1/m) * np.dot(dZ1, X.T)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {"dW1": dW1, "db1": db1,
             "dW2": dW2, "db2": db2,
             "dW3": dW3, "db3": db3}
    return grads


# ------------------------------
# مقداردهی ساختارهای Adam
# شامل میانگین‌گیری لحظه‌ای و میانگین مربعات
# ------------------------------
def initialize_adam(parameters):
    v, s = {}, {}
    for key in parameters:
        v["v_" + key] = np.zeros_like(parameters[key])
        s["s_" + key] = np.zeros_like(parameters[key])
    return v, s


In [ ]:
# ------------------------------
# تابع آموزش شبکه عصبی ۳ لایه با:
# - ۳۲ نورون در هر لایه مخفی
# - Adam Optimizer
# - ReLU
# - Xavier Initialization
# - تعداد epoch = 7000
# ------------------------------

def train_network_3layer_32(X, Y, num_epochs=7000, learning_rate=0.005,
                            n_h1=32, n_h2=32, beta1=0.9, beta2=0.999, epsilon=1e-8):

    n_x = X.shape[0]
    n_y = 1

    parameters = initialize_parameters_3layer(n_x, n_h1, n_h2, n_y)
    v, s = initialize_adam(parameters)

    losses = []
    t = 0

    for i in range(num_epochs):

        # فوروارد پروپاگیشن
        A3, cache = forward_propagation(X, parameters)

        # محاسبه هزینه Cross-Entropy
        loss = - np.mean(Y*np.log(A3 + 1e-8) + (1-Y)*np.log(1 - A3 + 1e-8))

        # بک‌پروپاگیشن
        grads = backward_propagation(X, Y, parameters, cache)

        # به‌روزرسانی پارامترها با Adam
        t += 1
        for key in parameters:

            v["v_" + key] = beta1 * v["v_" + key] + (1 - beta1) * grads["d" + key]
            s["s_" + key] = beta2 * s["s_" + key] + (1 - beta2) * (grads["d" + key]**2)

            v_corrected = v["v_" + key] / (1 - beta1**t)
            s_corrected = s["s_" + key] / (1 - beta2**t)

            parameters[key] -= learning_rate * v_corrected / (np.sqrt(s_corrected) + epsilon)

        if i % 200 == 0:
            print(f"Epoch {i}  Loss: {loss:.5f}")
            losses.append(loss)

    return parameters, losses


In [ ]:
# ------------------------------
# تابع پیش‌بینی خروجی شبکه
# خروجی نهایی: 0 یا 1
# ------------------------------

def predict_32(X, parameters):
    A3, _ = forward_propagation(X, parameters)
    return (A3 > 0.5).astype(int)


In [ ]:
# ------------------------------
# اجرای آموزش شبکه ۳ لایه با ۳۲ نورون
# ------------------------------

parameters_32, losses_32 = train_network_3layer_32(
    X_train,
    y_train,
    num_epochs=7000,
    learning_rate=0.005
)

# پیش‌بینی
train_pred = predict_32(X_train, parameters_32)
test_pred  = predict_32(X_test,  parameters_32)

# محاسبه دقت
train_acc = calculate_accuracy(train_pred, y_train)
test_acc  = calculate_accuracy(test_pred,  y_test)

print("\n---- Results for n_h = 32 ----")
print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)
